# Upload multiple files

This notebook shows you how to upload a folder of videos and images as assets, add them to a knowledge store, and wait for indexing. Uploading in bulk introduces two constraints that a single file does not:

- **Rate limits**: Uploads are rate-limited. A loop without pacing returns `429`.
- **Partial failures**: One file can fail while the rest succeed. Handle the failures separately.

# Key concepts

- **Asset**: Your uploaded content. Once created, you can reference the same asset across multiple operations without uploading the file again.
- **Knowledge store item**: An asset added to a knowledge store. The platform processes each item asynchronously. When processing finishes, the item is ready for downstream tasks.
- **Upload rate limit**: The number of uploads you can make per minute. Uploads share this limit across your account.

# Prerequisites

- To use the platform, you need an API key:
  1. If you don't have an account, [sign up](https://playground.twelvelabs.io/) for a free account.
  2. Go to the [API Keys](https://playground.twelvelabs.io/dashboard/api-keys) page.
  3. If you need to create a new key, select the **Create API Key** button. Enter a name and set the expiration period. The default is 12 months.
  4. Select the **Copy** icon next to your key to copy it to your clipboard.
- You've already created a knowledge store. See the [Create a knowledge store](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-knowledge-store) page for details.
- A folder of videos and images. Local video files support up to 200 MB, and images up to 32 MB. See the [Upload content](https://docs.twelvelabs.io/v1.3/agents/guides/upload-content) page for the other upload methods.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

# Configure the client

Set your API key, the knowledge store to add the assets to, and the folder to upload from.

In [ ]:
import concurrent.futures
import os
import time
from pathlib import Path

from twelvelabs import TwelveLabs
from twelvelabs.core.api_error import ApiError

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "<YOUR_API_KEY>")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "<YOUR_KNOWLEDGE_STORE_ID>")  # Replace with your knowledge store ID
MEDIA_DIR = Path(os.environ.get("TWELVELABS_MEDIA_DIR", "<YOUR_MEDIA_FOLDER>"))  # Folder of videos and images

client = TwelveLabs(api_key=API_KEY)

# Upload rate limit, in requests per minute. Free plan and Developer Tier 1.
# See https://docs.twelvelabs.io/v1.3/docs/get-started/rate-limits
UPLOAD_RPM = 60
# UPLOAD_RPM = 120  # Developer Tier 2
# UPLOAD_RPM = 240  # Developer Tier 3

UPLOAD_INTERVAL = 60 / UPLOAD_RPM  # Seconds to wait between uploads

# Rate limits

Uploads share a rate limit across your account. The Free plan and Developer Tier 1 allow 60 uploads per minute. Higher tiers allow more.

The setup cell sets `UPLOAD_RPM` to the Free plan value. Uncomment the tier that matches your plan.

This notebook uses two protections:

- **Spacing**: Wait between requests to stay under the limit.
- **Retrying**: Retry on `429` if another process shares the limit.

Adding items to a knowledge store is not rate-limited. Only the upload stage needs pacing. For the current limits, see the [Rate limits](https://docs.twelvelabs.io/v1.3/docs/get-started/rate-limits) page.

# Retry helper

Retry on transient errors (`429` rate limits and `5xx` server errors). Do not retry client errors (`400`, `401`, `403`, `404`). Fix the request instead.

In [ ]:
def with_retry(call, max_retries: int = 3, base_delay: int = 1):
    """Call an SDK method with exponential backoff on transient errors.

    Retries on 429 (rate limited) and 5xx (server error). Client errors
    (400, 401, 403, 404) are not retried - fix the request instead.

    Args:
        call: A zero-argument callable that performs the SDK request.
        max_retries: Maximum number of attempts before giving up.
        base_delay: Seconds to wait before the first retry, doubling each time.

    Returns:
        Whatever `call` returns.
    """
    for attempt in range(max_retries):
        try:
            return call()
        except ApiError as exc:
            status = exc.status_code
            if status == 429 or (status is not None and status >= 500):
                delay = base_delay * (2**attempt)
                print(f"  Received {status}, retrying in {delay}s...")
                time.sleep(delay)
                continue
            raise  # Client errors are not retryable
    raise RuntimeError(f"Failed after {max_retries} attempts")

# Upload the files

Upload every file in the folder, waiting between requests to stay under `UPLOAD_RPM`. This example collects failures instead of raising them, so a single file does not stop the batch.

The platform detects the kind of content automatically. You do not pass a type when uploading. Each asset returns a `file_type` field containing the MIME type the platform detected, such as `video/mp4` or `image/jpeg`.

In [ ]:
MEDIA_SUFFIXES = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".jpg", ".jpeg", ".png"}

files = sorted(p for p in MEDIA_DIR.iterdir() if p.suffix.lower() in MEDIA_SUFFIXES)
print(f"Found {len(files)} files in {MEDIA_DIR}")

uploaded = []  # (path, asset) for each success
failed = []  # (path, error) for each failure

for path in files:
    try:
        asset = with_retry(
            lambda p=path: client.assets.create(method="direct", file=open(p, "rb"))
        )
        uploaded.append((path, asset))
        print(f"  {path.name} -> {asset.id}  ({asset.file_type})")
    except ApiError as exc:
        failed.append((path, exc))
        print(f"  {path.name} -> FAILED: {exc.status_code}")

    time.sleep(UPLOAD_INTERVAL)  # Stay under the upload rate limit

print(f"\nUploaded {len(uploaded)}, failed {len(failed)}")

# Check the status of the assets

The platform processes uploads asynchronously. Each asset must reach the `ready` status before you add it to a knowledge store.

In [ ]:
def wait_for_asset(asset_id: str, interval: int = 5, timeout: int = 300):
    """Poll an asset until it is ready or failed.

    Args:
        asset_id: The asset to poll.
        interval: Seconds between poll attempts.
        timeout: Seconds to wait before giving up.

    Returns:
        The asset once it reaches the `ready` status.

    Raises:
        Exception: If the asset reaches the `failed` status.
        TimeoutError: If the asset is not ready before the timeout.
    """
    elapsed = 0
    while elapsed < timeout:
        asset = client.assets.retrieve(asset_id=asset_id)
        if asset.status == "ready":
            return asset
        if asset.status == "failed":
            raise Exception(f"Asset failed: {asset_id}")
        time.sleep(interval)
        elapsed += interval
    raise TimeoutError(f"Asset {asset_id} not ready after {timeout}s")


ready_assets = []
for path, asset in uploaded:
    try:
        ready_assets.append((path, wait_for_asset(asset.id)))
        print(f"  {path.name}: ready")
    except (Exception, TimeoutError) as exc:
        failed.append((path, exc))
        print(f"  {path.name}: {exc}")

print(f"\n{len(ready_assets)} assets ready")

# Add the assets to the knowledge store

Videos and images differ in one parameter. Set `asset_type` to `image` when the asset is an image. It defaults to `video`.

You do not need to track the kind yourself. The `file_type` field on each asset records what the platform detected during upload.

In [ ]:
item_ids = []

for path, asset in ready_assets:
    # The platform detected the kind on upload; declare it when adding to the store.
    asset_type = "image" if asset.file_type.startswith("image/") else "video"

    item = client.knowledge_store_items.create(
        knowledge_store_id=STORE_ID,
        asset_id=asset.id,
        asset_type=asset_type,
    )
    item_ids.append(item.id)
    print(f"  {path.name} -> item {item.id}  ({asset_type})")

print(f"\nAdded {len(item_ids)} items")

# Check the status of the knowledge store items

Indexing runs asynchronously and usually takes longer than the asset processing above. Poll the items at a longer interval. This example polls every item in parallel.

In [ ]:
def wait_for_item(item_id: str, interval: int = 10, timeout: int = 600):
    """Poll a knowledge store item until it is ready or failed."""
    elapsed = 0
    while elapsed < timeout:
        item = client.knowledge_store_items.retrieve(
            knowledge_store_id=STORE_ID, item_id=item_id
        )
        if item.status == "ready":
            return item
        if item.status == "failed":
            raise Exception(f"Indexing failed: {item_id}")
        time.sleep(interval)
        elapsed += interval
    raise TimeoutError(f"Item {item_id} not ready after {timeout}s")


with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(wait_for_item, iid): iid for iid in item_ids}
    for future in concurrent.futures.as_completed(futures):
        item_id = futures[future]
        try:
            future.result()
            print(f"  {item_id}: ready")
        except Exception as exc:
            print(f"  {item_id}: {exc}")

print("\nIndexing complete. The knowledge store is ready to query.")

# Handle partial failures

The `failed` list holds each file that failed and the error that caused it. Check the status code before you retry. A `400` means the request is invalid, so a retry returns the same error.

In [ ]:
for path, exc in failed:
    status = getattr(exc, "status_code", None)
    if status == 400:
        print(f"  {path.name}: rejected by the platform ({exc}). Check format and size.")
    else:
        print(f"  {path.name}: {exc}. Safe to retry.")

if not failed:
    print("No failures.")

# Common pitfalls

- **Uploading without pacing**: Uploads are rate-limited. A loop without a wait returns `429`. Adding items to a knowledge store is not rate-limited, so only the upload stage needs pacing.
- **Omitting `asset_type` for images**: The parameter defaults to `video`. Derive it from the `file_type` field of the asset instead of tracking it yourself.
- **Stopping at the first error**: Collect the failures and continue. One unsupported file does not affect the rest of the folder.
- **Polling assets and items at the same interval**: Indexing takes longer than upload processing. Use a longer interval for items.
- **Retrying client errors**: Only `429` and `5xx` are transient. Retrying a `400` returns the same error.

# Next steps

- [Uploading content](uploading_content.ipynb): Upload a single video or image, and enable HLS or thumbnails.
- [Building knowledge stores](building_knowledge_stores.ipynb): Configure ingestion and store options.
- [Querying](querying.ipynb): Ask questions about your videos and images.
- [Error handling](error_handling.ipynb): Retry strategy and polling helpers in depth.

**API Reference:** [POST /assets](https://docs.twelvelabs.io/v1.3/api-reference/upload-content/direct-uploads/create)